## Hierarchical Feature Engineering

This notebook explores of feature engineering and modelling for hierarchical time series. The provided dataset has four groups (hyperioncode) with a supplier and part number. Each group has between 10,000 to 40,000 observations. In the forecast, we have many parts and suppliers that have low counts and sporadic demand.

In [0]:
import numpy as np
import pandas as pd

from datasetsforecast.hierarchical import HierarchicalData, HierarchicalInfo
group_name = 'Traffic'
group = HierarchicalInfo.get_group(group_name)
y_hier, s_df, tags = HierarchicalData.load('./data', group_name)
s_df = s_df.reset_index(names="unique_id")
y_hier['ds'] = pd.to_datetime(y_hier['ds'])

This data is hierarchical, but with sparse observations. We need to normalize the data and make a proper hierarchical dataframe, ensuring that each group / subgroup is accounted for.

In [0]:
y_hier

Next thing we need to do is resample and fill missing values with zero - pretty much every method we use will require regular time series data.

In [0]:
from utilsforecast.preprocessing import fill_gaps
y_filled = (
    fill_gaps(y_hier, freq='D', start='global', end='global')
    .fillna(0)
    .sort_values(by=['unique_id', 'ds'])
)
y_filled

In [0]:
from utilsforecast.plotting import plot_series
plot_series(y_filled, engine='plotly', ids=['Total', 'y11', 'Bottom9', 'Bottom85'])

Let's setup a problem to predict the next 3 months of demand. Let's assume intermittent series and use a naive, ARIMA, and ADIDA (intermittent model). This training can also be distributed using Spark!

https://nixtlaverse.nixtla.io/statsforecast/docs/distributed/spark.html

https://nixtlaverse.nixtla.io/statsforecast/docs/experiments/prophet_spark_m5.html

In [0]:
# Let's train some baseline moodels
from statsforecast.models import Naive, ADIDA, AutoARIMA
from statsforecast.core import StatsForecast

# Split train/test sets
y_test_df  = y_filled.groupby('unique_id', as_index=False).tail(14)
y_train_df = y_filled.drop(y_test_df.index)

# Compute base Naive and ADIDA (intermittent model) predictions
fcst = StatsForecast(models=[
    Naive(),
    AutoARIMA(),
    ADIDA()
    ], freq='D', n_jobs=-1)

In [0]:
y_hat_df = fcst.forecast(df=y_train_df, h=14, fitted=True)

# This is necessary for the reconciliation step
y_fitted_df = fcst.forecast_fitted_values()

In [0]:
y_hat_df

In [0]:
plot_series(y_filled, y_hat_df, engine='plotly', ids=['Total', 'y11', 'Bottom9', 'Bottom85'])

In [0]:
# Now let's evaluate our forecasts
from utilsforecast.evaluation import evaluate
from utilsforecast.losses import mae, rmse, mape

eval_df = evaluate(
    y_test_df.merge(y_hat_df, on=['unique_id', 'ds'], how='inner'),
    train_df=y_train_df,
    metrics=[mae, rmse, mape],
    models=['Naive', 'AutoARIMA', 'ADIDA']
)
eval_df

In [0]:
summary = eval_df.drop(columns='unique_id').groupby('metric').mean().reset_index()
summary

One of the features of hiearchical time series forecasting is reconciliation of forecasts. This is done to ensure that the sum of the forecasts for each group is equal to the forecast for the overall group. This may or may not be important depending on your use case.

In [0]:
from hierarchicalforecast.methods import BottomUpSparse, BottomUp, MinTrace
from hierarchicalforecast.core import HierarchicalReconciliation

reconcilers = [BottomUp(), MinTrace(method='mint_shrink')]

hrec = HierarchicalReconciliation(reconcilers=reconcilers)

Y_rec_df = hrec.reconcile(Y_hat_df=y_hat_df.fillna(0), 
                          Y_df=y_fitted_df.fillna(0),
                          S=s_df, tags=tags)
Y_rec_df.groupby('unique_id').head(2)

Now let's get into feature engineering. I'm using Nixtla here because they have native Spark and Ray support and play very well at scale with Databricks. As we do feature engineering, we move from univariate classical time series models to local machine learning models and leverage the MLForecast library.

We have to be careful about our transforms and lags because we will drop a lot of the series that don't have sufficient data. So we will limit our lags to less than 6 months. This can be challenging because we can't capture yearly seasonality.

In [0]:
from matplotlib import pyplot as plt
y_filled.groupby('unique_id').count().y.hist(figsize=(4,3))
plt.title('Series Lengths in Months')
#plt.axvline(x=6, color='red', linestyle='-')
plt.show()

This gives us univariate features that lend themselves well to local machine learning models.

In [0]:
from mlforecast import MLForecast
from mlforecast.lag_transforms import ExpandingMean, RollingMean
from mlforecast.target_transforms import Differences

fcst = MLForecast(
    models=[],
    freq='MS',
    lags=[3, 6],
    lag_transforms={
        3: [ExpandingMean()],
        6: [RollingMean(window_size=3)]
    },
    date_features=['quarter', 'daysinmonth'],
    target_transforms=[Differences([1])],
)

feats = fcst.preprocess(df=y_filled)
display(feats.sample(5))

We can also seasonal decompostion. To do this, we will need to drop series with less than 12 months of data since we need enough data to properly estimate seasonality. These can then be used as exogenous features in ML models.

https://nixtlaverse.nixtla.io/statsforecast/docs/how-to-guides/generating_features.html


In [0]:
series_lengths = y_train_df.groupby('unique_id').size()
valid_series = series_lengths[series_lengths >= 12].index
y_train_df_long = y_train_df[y_train_df['unique_id'].isin(valid_series)]
y_train_df_long

In [0]:
from statsforecast.feature_engineering import mstl_decomposition
from statsforecast.models import AutoARIMA, MSTL

transformed_df, X_df = mstl_decomposition(
    y_train_df_long, 
    model=MSTL(season_length=3), 
    freq='MS', 
    h=3
    )
display(X_df.sample(5))

Without additional exogenous features, we can only leverage lags, rolling means, decomposition information, and categorical features based on the unique_id (we will only use the hyperioncode for now). But we can use these features to train a local machine learning model, or a global model.

In [0]:
import lightgbm as lgb
from sklearn.neural_network import MLPRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression

models={
        'lightgbm': lgb.LGBMRegressor(verbosity=-1),
        'knn': KNeighborsRegressor(),
        'mlp': MLPRegressor(),
        'linear': LinearRegression()
    }

fcst = MLForecast(
    models=models,
    freq='D',
    lags=[3, 6],
    lag_transforms={
        3: [ExpandingMean(), RollingMean(window_size=3)],
        6: [ExpandingMean(), RollingMean(window_size=6)]
    },
    date_features=['quarter', 'daysinmonth'],
    target_transforms=[Differences([1])]
)

Let's run cross validation, because it is the right thing to do.

In [0]:
fcst.fit(
    df=y_train_df,
    static_features=[]
)

In [0]:
forecast_df = fcst.predict(14)
plot_series(y_filled, forecast_df, engine='plotly', ids=['Total', 'y11', 'Bottom9', 'Bottom85'])

Interestingly, the ML models actually perform quite a bit worse than the traditional models.

In [0]:
eval_df = evaluate(
    forecast_df.merge(y_test_df),
    metrics=[mae, rmse, mape],
    models=['lightgbm', 'knn', 'mlp', 'linear']
)
summary = eval_df.drop(columns='unique_id').groupby('metric').mean().reset_index()
summary